In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("Churn.csv")

In [2]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### EDA

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
df.isna().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [5]:
df['TotalCharges'] = df['TotalCharges'].replace(" ", np.nan)
df['TotalCharges'] = df['TotalCharges'].astype(float)

In [6]:
pd.crosstab(df['Contract'],df['Churn'])

Churn,No,Yes
Contract,,
Month-to-month,2220,1655
One year,1307,166
Two year,1647,48


In [7]:
pd.crosstab(df['InternetService'], df['Churn'])

Churn,No,Yes
InternetService,,
DSL,1962,459
Fiber optic,1799,1297
No,1413,113


In [8]:
pd.crosstab(df['PaymentMethod'], df['Churn'])

Churn,No,Yes
PaymentMethod,,
Bank transfer (automatic),1286,258
Credit card (automatic),1290,232
Electronic check,1294,1071
Mailed check,1304,308


In [9]:
df.groupby('Churn')['MonthlyCharges'].mean()

Churn
No     61.265124
Yes    74.441332
Name: MonthlyCharges, dtype: float64

### 📊 Key Business Insights after EDA

- Customers with month-to-month contracts show significantly higher churn.

- Fiber optic internet users have higher churn compared to other services.

- Customers paying via electronic check are more likely to churn.

- Higher monthly charges are associated with increased churn risk.

## ML Applied

In [10]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [11]:
df.Churn.value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [12]:
df = df.drop('customerID', axis=1)
df_clean = df.copy()

In [13]:
df_clean = pd.get_dummies(df_clean, drop_first=True)

In [14]:
df_clean.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,False,True,False,False,True,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,True,False,False,True,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,True,False,False,True,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,True,False,False,False,True,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,1,False,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False


In [15]:
df_clean = df_clean.dropna()

In [16]:
X = df_clean.drop('Churn', axis=1)
y = df_clean['Churn']

In [17]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [19]:
X_train.shape, X_test.shape

((5625, 30), (1407, 30))

### Logistic Regression

In [20]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

c:\Users\jayan_c3r1z\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [21]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy on test:", accuracy)


Accuracy on test: 0.7874911158493249


In [22]:
y_train_pred = model.predict(X_train)
train_accuracy = accuracy_score(y_train, y_train_pred)

print("Train Accuracy:", train_accuracy)
print("Test Accuracy:", accuracy)

Train Accuracy: 0.8088888888888889
Test Accuracy: 0.7874911158493249


In [23]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, y_pred)

array([[915, 118],
       [181, 193]])

In [24]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1033
           1       0.62      0.52      0.56       374

    accuracy                           0.79      1407
   macro avg       0.73      0.70      0.71      1407
weighted avg       0.78      0.79      0.78      1407



## 🤖 Model Summary

- Model used: Logistic Regression  
- Problem type: Binary Classification  
- Evaluation metric: Accuracy  

The model was trained to predict customer churn based on historical data.

### Recall is important in churn because missing a churn customer means losing revenue

#### Random forest

In [25]:
from sklearn.ensemble import RandomForestClassifier

I used Random Forest to capture non-linear patterns and improve performance over Logistic Regression.

In [26]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

In [27]:
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, y_pred_rf)
print("Random Forest Test Accuracy:", rf_accuracy)

Random Forest Test Accuracy: 0.7853589196872779


In [28]:
print("Logistic Regression Accuracy:", accuracy)
print("Random Forest Accuracy:", rf_accuracy)

Logistic Regression Accuracy: 0.7874911158493249
Random Forest Accuracy: 0.7853589196872779


In [29]:
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1033
           1       0.63      0.48      0.54       374

    accuracy                           0.79      1407
   macro avg       0.73      0.69      0.70      1407
weighted avg       0.77      0.79      0.78      1407



### Comparsion Between Logistic Regression and RF 

In [30]:
print(classification_report(y_test, y_pred))
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1033
           1       0.62      0.52      0.56       374

    accuracy                           0.79      1407
   macro avg       0.73      0.70      0.71      1407
weighted avg       0.78      0.79      0.78      1407

              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1033
           1       0.63      0.48      0.54       374

    accuracy                           0.79      1407
   macro avg       0.73      0.69      0.70      1407
weighted avg       0.77      0.79      0.78      1407



Logistic Regression is actually better here

Because:

    It catches more churn customers (52% vs 48%).

    Accuracy is same.

    Precision is similar.


Compared Logistic Regression and Random Forest using multiple metrics. While both models had similar accuracy (~79%), Logistic Regression achieved higher recall for churn customers (52% vs 48%). Since identifying churn customers is critical for business, I selected Logistic Regression as the better model.


### Considering Data Imbalance

In [31]:
model_balanced = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

model_balanced.fit(X_train, y_train)

c:\Users\jayan_c3r1z\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [32]:
y_pred_bal = model_balanced.predict(X_test)

In [33]:
y_train_pred_bal = model_balanced.predict(X_train)
train_accuracy_bal = accuracy_score(y_train, y_train_pred_bal)

y_test_pred_bal = model_balanced.predict(X_test)
test_accuracy_bal = accuracy_score(y_test, y_test_pred_bal)

print("Train Accuracy:", train_accuracy_bal)
print("Test Accuracy:", test_accuracy_bal)

Train Accuracy: 0.7548444444444444
Test Accuracy: 0.7341862117981521


In [34]:
print(classification_report(y_test, y_test_pred_bal))

              precision    recall  f1-score   support

           0       0.91      0.71      0.80      1033
           1       0.50      0.79      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.71      1407
weighted avg       0.80      0.73      0.75      1407



In [35]:
rf_balanced = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)

rf_balanced.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [36]:
y_pred_rf_bal = rf_balanced.predict(X_test)
print(classification_report(y_test, y_pred_rf_bal))

              precision    recall  f1-score   support

           0       0.82      0.91      0.86      1033
           1       0.63      0.45      0.53       374

    accuracy                           0.78      1407
   macro avg       0.73      0.68      0.69      1407
weighted avg       0.77      0.78      0.77      1407



Massively improved recall:

52% → 79%

Meaning:

Earlier: missing many churn customers ❌

Now: catching most churn customers ✅

Sacrifice?

Precision dropped: More false positives


Meaning:

Some customers predicted as churn won’t actually churn

### Business Interpretation 

Which is worse?

❌ Missing churn customer

Lose revenue

Customer gone

❌ False alarm (predict churn wrongly)

Offer discount unnecessarily


Clearly:
Missing churn is worse

### Feature engineering

In [37]:
df_clean = df.copy()
df_clean = df_clean.dropna()

In [38]:
df_clean['tenure_group'] = pd.cut(
    df_clean['tenure'],
    bins=[0, 12, 24, 48, 60, 100],
    labels=['0-1yr', '1-2yr', '2-4yr', '4-5yr', '5+yr']
)

In [39]:
df_clean.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn', 'tenure_group'],
      dtype='object')

In [40]:
services = ['PhoneService', 'InternetService', 'OnlineSecurity',
            'OnlineBackup', 'DeviceProtection', 'TechSupport',
            'StreamingTV', 'StreamingMovies']

df_clean['TotalServices'] = df_clean[services].apply(lambda x: (x != 'No').sum(), axis=1)

In [41]:
df_clean = df_clean.dropna()

df_clean = pd.get_dummies(df_clean, drop_first=True)

In [42]:
X = df_clean.drop('Churn', axis=1)
y = df_clean['Churn']



In [43]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [44]:
model_balanced = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

model_balanced.fit(X_train, y_train)

c:\Users\jayan_c3r1z\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [45]:
y_train_pred_bal = model_balanced.predict(X_train)
train_accuracy_bal = accuracy_score(y_train, y_train_pred_bal)

y_test_pred_bal = model_balanced.predict(X_test)
test_accuracy_bal = accuracy_score(y_test, y_test_pred_bal)

print("Train Accuracy:", train_accuracy_bal)
print("Test Accuracy:", test_accuracy_bal)

Train Accuracy: 0.7539555555555556
Test Accuracy: 0.7334754797441365


In [46]:
print(classification_report(y_test, y_test_pred_bal))

              precision    recall  f1-score   support

           0       0.91      0.71      0.80      1033
           1       0.50      0.80      0.62       374

    accuracy                           0.73      1407
   macro avg       0.70      0.76      0.71      1407
weighted avg       0.80      0.73      0.75      1407



In [47]:
df_clean.groupby('TotalServices')['Churn'].mean()

TotalServices
1    0.437500
2    0.515818
3    0.434739
4    0.346782
5    0.272642
6    0.220606
7    0.087573
8    0.057915
Name: Churn, dtype: float64

While the improvement in recall is modest (~1%), it confirmed that customer engagement has some influence on churn. This demonstrates an iterative feature engineering approach rather than relying on assumptions. 

In [48]:

feature_importance = pd.Series(
    model_balanced.coef_[0],
    index=X.columns
)

In [49]:
feature_importance = feature_importance.sort_values(key=abs, ascending=False)
feature_importance.head(10)

Contract_Two year                -1.456863
Contract_One year                -0.795629
MultipleLines_No phone service    0.488793
tenure_group_1-2yr               -0.476426
OnlineSecurity_Yes               -0.356770
TechSupport_Yes                  -0.330847
PhoneService_Yes                 -0.311265
InternetService_Fiber optic       0.289843
PaymentMethod_Electronic check    0.284970
SeniorCitizen                     0.273862
dtype: float64

In [50]:
feature_importance

Contract_Two year                       -1.456863
Contract_One year                       -0.795629
MultipleLines_No phone service           0.488793
tenure_group_1-2yr                      -0.476426
OnlineSecurity_Yes                      -0.356770
TechSupport_Yes                         -0.330847
PhoneService_Yes                        -0.311265
InternetService_Fiber optic              0.289843
PaymentMethod_Electronic check           0.284970
SeniorCitizen                            0.273862
PaperlessBilling_Yes                     0.272304
tenure_group_4-5yr                       0.238742
StreamingMovies_Yes                      0.206854
StreamingTV_Yes                          0.202568
tenure_group_2-4yr                      -0.197685
Dependents_Yes                          -0.195542
TotalServices                           -0.136310
tenure_group_5+yr                        0.135152
MultipleLines_Yes                        0.123209
PaymentMethod_Mailed check              -0.080665


In [51]:
top_positive = feature_importance.sort_values(ascending=False).head(5)
top_negative = feature_importance.sort_values().head(5)

print(top_positive,"\n\n", top_negative)

MultipleLines_No phone service    0.488793
InternetService_Fiber optic       0.289843
PaymentMethod_Electronic check    0.284970
SeniorCitizen                     0.273862
PaperlessBilling_Yes              0.272304
dtype: float64 

 Contract_Two year    -1.456863
Contract_One year    -0.795629
tenure_group_1-2yr   -0.476426
OnlineSecurity_Yes   -0.356770
TechSupport_Yes      -0.330847
dtype: float64


## 📊 Feature Importance Insights

### 🔺 Factors Increasing Churn Risk

- Customers without phone service but having multiple line indicators show higher churn tendency, possibly indicating inconsistent or low engagement.

- Customers using Fiber Optic internet services are more likely to churn. This may be due to higher pricing or service-related dissatisfaction.

- Customers paying via Electronic Check have higher churn rates, suggesting this payment method is associated with less stable or less committed customers.

---

### 🔻 Factors Reducing Churn Risk

- Customers with long-term contracts (One-year and Two-year) are significantly less likely to churn. This highlights the importance of contract-based retention strategies.

- Customers in early tenure stages (1–2 years) show lower churn compared to very new customers, indicating that retention improves after initial onboarding.

- Customers who have opted for value-added services like Online Security and Tech Support are less likely to churn, suggesting higher engagement and satisfaction.

- Customers with higher total number of services (TotalServices) also show reduced churn, reinforcing that more engaged customers are more loyal.

---

### 🧠 Business Interpretation

- Contract duration is the strongest driver of customer retention.

- Service engagement plays a critical role—customers using more services and add-ons are less likely to leave.

- Certain segments (Fiber optic users, electronic check users) should be targeted for proactive retention strategies.

---

### 🎯 Actionable Recommendations

- Encourage customers to move to long-term contracts through offers or discounts.

- Promote bundled services (security, tech support) to increase engagement.

- Identify high-risk groups (fiber users, electronic check users) and design targeted retention campaigns.

## Saving Model

In [52]:
import pickle

In [53]:
# Storing model; feature ordering(sequence) for reuse and deployment under churn_model filename 
pickle.dump(model_balanced, open('churn_model.pkl', 'wb'))

pickle.dump(X.columns, open('churn_model_columns.pkl', 'wb'))

In [54]:
''' In case we want to use this model and predict for new samples'''

'''model = pickle.load(open('churn_model.pkl', 'rb'))

sample = X_test.iloc[0].values.reshape(1, -1)
prediction = model.predict(sample)

print("Churn Prediction:", prediction)'''

'model = pickle.load(open(\'churn_model.pkl\', \'rb\'))\n\nsample = X_test.iloc[0].values.reshape(1, -1)\nprediction = model.predict(sample)\n\nprint("Churn Prediction:", prediction)'

## Cross Validation

In [55]:
from sklearn.model_selection import cross_val_score

model = LogisticRegression(max_iter=1000, class_weight='balanced')

scores = cross_val_score(model, X, y, cv=5, scoring='recall')

print("Recall scores:", scores)
print("Average Recall:", scores.mean())

c:\Users\jayan_c3r1z\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\jayan_c3r1z\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
 

Recall scores: [0.83957219 0.81016043 0.80160858 0.78074866 0.75935829]
Average Recall: 0.798289630256197


c:\Users\jayan_c3r1z\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [56]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced'))
])

scores = cross_val_score(pipeline, X, y, cv=5, scoring='recall')

In [58]:
print("Recall scores:", scores)
print("Average Recall:", scores.mean())

Recall scores: [0.83957219 0.81550802 0.80428954 0.77540107 0.76203209]
Average Recall: 0.7993605826439765


## Regularization


In [59]:
model_l2 = LogisticRegression(
    penalty='l2',
    max_iter=1000,
    class_weight='balanced'
)

model_l1 = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    max_iter=1000,
    class_weight='balanced'
)

In [60]:
model_l2.fit(X_train, y_train)
model_l1.fit(X_train, y_train)

c:\Users\jayan_c3r1z\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\jayan_c3r1z\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.htm

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l1'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [61]:
y_pred_l2 = model_l2.predict(X_test)
y_pred_l1 = model_l1.predict(X_test)

In [62]:
from sklearn.metrics import accuracy_score

print("L2 Accuracy:", accuracy_score(y_test, y_pred_l2))
print("L1 Accuracy:", accuracy_score(y_test, y_pred_l1))

L2 Accuracy: 0.7334754797441365
L1 Accuracy: 0.7334754797441365


In [63]:
from sklearn.metrics import recall_score

print("L2 Recall:", recall_score(y_test, y_pred_l2))
print("L1 Recall:", recall_score(y_test, y_pred_l1))

L2 Recall: 0.8021390374331551
L1 Recall: 0.8021390374331551


In [64]:
from sklearn.metrics import classification_report

print("L2 Report:\n", classification_report(y_test, y_pred_l2))
print("L1 Report:\n", classification_report(y_test, y_pred_l1))

L2 Report:
               precision    recall  f1-score   support

           0       0.91      0.71      0.80      1033
           1       0.50      0.80      0.62       374

    accuracy                           0.73      1407
   macro avg       0.70      0.76      0.71      1407
weighted avg       0.80      0.73      0.75      1407

L1 Report:
               precision    recall  f1-score   support

           0       0.91      0.71      0.80      1033
           1       0.50      0.80      0.62       374

    accuracy                           0.73      1407
   macro avg       0.70      0.76      0.71      1407
weighted avg       0.80      0.73      0.75      1407



In [65]:
import pandas as pd

coef_l2 = pd.Series(model_l2.coef_[0], index=X.columns)
coef_l1 = pd.Series(model_l1.coef_[0], index=X.columns)

print("Top L2 Features:\n", coef_l2.sort_values(ascending=False).head())
print("Top L1 Features:\n", coef_l1.sort_values(ascending=False).head())

Top L2 Features:
 MultipleLines_No phone service    0.488793
InternetService_Fiber optic       0.289843
PaymentMethod_Electronic check    0.284970
SeniorCitizen                     0.273862
PaperlessBilling_Yes              0.272304
dtype: float64
Top L1 Features:
 InternetService_Fiber optic       0.647026
MultipleLines_No phone service    0.571090
tenure_group_4-5yr                0.359332
PaymentMethod_Electronic check    0.299407
StreamingMovies_Yes               0.292746
dtype: float64


In [66]:
print("L1 zero coefficients:", (coef_l1 == 0).sum())
print("L2 zero coefficients:", (coef_l2 == 0).sum())

L1 zero coefficients: 7
L2 zero coefficients: 0


Both L1 and L2 regularization gave similar performance, indicating that the model is robust and not highly sensitive to regularization choice.

L2 top features:
MultipleLines_No phone service
Fiber optic
Payment method


L1 top features:
Fiber optic (stronger weight)
tenure_group
Streaming

Both L1 and L2 performed similarly in terms of recall and accuracy. However, L1 reduced model complexity by eliminating some features, making it useful for feature selection. L2 retained all features but shrank their impact. Since performance was similar, L1 could be preferred for simplicity and interpretability.